In [1]:
from demande.training.model_building.generate_model import generate_model
from demande.training.iterate import iterate
from demande.visualizations.plot_probability import plot_probability
from demande.utils.metrics_calculator import calculate_metrics
from demande.visualizations.plot_graph_density import plot_graph_density
from demande.utils.grid_definition import define_grid
from demande.dataset_utils.load_dataset import load_dataset
from demande.utils.get_current_day import get_current_day
import tensorflow as tf
import matplotlib.pyplot as plt
import logging
from demande.training.get_probability import get_probability
from demande.mlflow_utils import mlflow_wrapper



ModuleNotFoundError: No module named 'demande'

In [ ]:
train_size = 0.7
test_size = 0.3

In [ ]:
setting = {
    "z_base_lr": 1e-3,
    "z_end_lr": 1e-7,
    "z_max_epochs": int(50),
    "z_decay_steps": int(50),
    "z_dataset_size": 40000,
    "z_weight_decay": 1e-4, 
    "z_batch_size": 64,
    "z_train_size": 40000,
    "z_test_size": 1000,
    "z_power": 0.5,
    "z_algorithm": f"{algorithm}",
    "z_experiment": f"{database}_{algorithm}",
    "z_dataset": f"{database}",
    "experiments_folder": experiments_folder,
    "logs": experiments_folder + "logs/",
    "models": experiments_folder + "models/",
    "datasets": experiments_folder + "datasets/",
    "sufix_name_experiment" : f"{database}_{algorithm}",
    "z_dimension": 2,
    "z_run_name": f"{database}_{algorithm}",
    "z_plot_graph_density": True,
    "z_name_of_experiment": 'conditional-density-experiment',
    "z_random_search": True,
    #"z_random_search": False,
    "z_random_search_random_state": 1,
    "z_random_search_iter": 30,
    "z_step": "train", 
    "z_adaptive_epochs": 100,
    "z_adaptive_learning_rate": 0.0001,
    "z_adaptive_batch_size": 64
}

In [ ]:

dimension = setting["z_dimension"]
batch_size = setting["z_batch_size"]
dataset = setting["z_dataset"]
print(setting)


X_train, X_train_density, X_test, X_test_densities = load_dataset(dataset, train_size, test_size, dimension)


train_dataset = tf.data.Dataset.from_tensor_slices(X_train)
batched_train_data = train_dataset.batch(batch_size)
print(f'##----------#----------------------#---------------#')
print(f'Train 2D data set shape is:{X_train.shape} ')
print(f'Test 2D data set shape is:{X_test.shape} and their densities shape is {X_test_densities.shape}')
print(f'##----------#----------------------#---------------#')

plt.figure()
plt.axes(frameon = 0)
plt.grid()
plt.scatter(X_test[:,0], X_test[:,1], c = X_test_densities, alpha=.3,s = 3, linewidths= 0.0000001)
plt.colorbar()
#plt.savefig(f"reports/original_data_2d.png")
plt.show()

model = generate_model(setting)

polinomial_decay = tf.keras.optimizers.schedules.PolynomialDecay(setting["z_base_lr"], \
        setting["z_decay_steps"], setting["z_end_lr"], power=setting["z_power"])
opt = tf.keras.optimizers.Adam(learning_rate=polinomial_decay)  # optimizer

checkpoint = tf.train.Checkpoint(optimizer=opt, model=model)

iterate(model, setting["z_max_epochs"], batched_train_data, X_train, X_test, opt, checkpoint, checkpoint_prefix, setting["z_algorithm"], mlflow, setting)

if("verbose" in setting and setting["verbose"]): f"Iterate finished {i}"

if X_train.shape[1] == 2:
    x_grid, y_grid, xy_plot_grid, x_number, y_number = define_grid(X_train)
    probability = get_probability(model, xy_plot_grid, setting["z_algorithm"], setting["z_dimension"], setting["z_sigma"])
    print(int(x_number), int(y_number))
    plot_probability(probability, x_grid, y_grid, round(x_number), round(y_number), path_name=f'reports/grid_mesh_probability.png')
    mlflow_wrapper.log_artifact(mlflow, f'reports/grid_mesh_probability.png')

estimated_density = get_probability(model, X_test, setting["z_algorithm"], setting["z_dimension"], setting["z_sigma"])

metrics = calculate_metrics(X_test_densities, estimated_density)



